## Learning within the same participant and test on its unseen trial

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import resample

# ========================================================================
# 1. UNITY MAPPING
# ========================================================================
dataset_dir = Path(r"D:\MUedit-main\dataset_sincronizzato_ML")
save_dir = Path(r"D:\MUedit-main\modelli_salvati")

SOGGETTO_TARGET = "subject_2"
TRIAL_TEST_SELEZIONATO = 4
seq_length = 40
fs_originale = 2000.0
fs_target = 200.0
fs_omogenea = fs_target

flexion_channels = {
    "Thumb_MCP": 60.0, "Thumb_IP": 80.0,
    "Index_MCP": 75.0, "Index_PIP": 95.0,
    "Middle_MCP": 75.0, "Middle_PIP": 100.0,
    "Ring_MCP": 75.0, "Ring_PIP": 100.0,
    "Little_MCP": 75.0, "Little_PIP": 95.0,
}
abduction_channels = {
    "Thumb_Index_abd": 30.0, "Index_Middle_abd": 20.0,
    "Middle_Ring_abd": 20.0, "Ring_Little_abd": 25.0,
}
channel_names = [
    "Thumb_MCP", "Thumb_IP", "Thumb_Index_abd", 
    "Index_MCP", "Index_PIP", "Index_Middle_abd", 
    "Middle_MCP", "Middle_PIP", "Middle_Ring_abd", 
    "Ring_MCP", "Ring_PIP", "Ring_Little_abd", 
    "Little_MCP", "Little_PIP"
]

def convert_to_unity(y_data, fs=200.0):
    n_samples = y_data.shape[0]
    time_s = np.arange(n_samples) / fs
    df = pd.DataFrame({"time_s": time_s})
    pred_df = pd.DataFrame(y_data, columns=channel_names)
    for ch in channel_names:
        vals = pred_df[ch].values
        if ch in flexion_channels:
            df[ch] = vals * flexion_channels[ch]
        elif ch in abduction_channels:
            df[ch] = (vals - 0.5) * abduction_channels[ch]
        else:
            df[ch] = vals * 90.0
    return df

def load_and_prepare_data(file_path):
    data = np.load(file_path)
    X, y = data['X'], data['y']
    
    num_target = int(len(X) * (fs_target / fs_originale))
    X = resample(X, num_target, axis=0)
    y = resample(y, num_target, axis=0)
    
    X[X < 0] = 0
    y = np.clip(y, 0.0, 1.0)
    
    X_scaled = X * 1000
    X_mean, X_std = X_scaled.mean(axis=0), X_scaled.std(axis=0)
    X_norm = (X_scaled - X_mean) / (X_std + 1e-8)
    
    sequences, targets = [], []
    for t in range(len(X_norm) - seq_length):
        sequences.append(X_norm[t:t+seq_length])
        targets.append(y[t+seq_length-1])
        
    return sequences, targets

# ========================================================================
# 2. DEFINE MODEL AND LOSS FUNCTION
# ========================================================================
class SynergyLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=2, output_size=14, dropout=0.3):
        super(SynergyLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32), 
            nn.ReLU(), 
            nn.Dropout(dropout), 
            nn.Linear(32, output_size)
        )
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

class SmoothMSELoss(nn.Module):
    def __init__(self, lambda_smooth=0.1):
        super(SmoothMSELoss, self).__init__()
        self.mse = nn.MSELoss()
        self.lambda_smooth = lambda_smooth

    def forward(self, y_pred, y_true):
        mse_loss = self.mse(y_pred, y_true)
        if y_pred.shape[0] > 1:
            smoothness_loss = torch.mean((y_pred[1:] - y_pred[:-1])**2)
        else:
            smoothness_loss = 0.0
        return mse_loss + self.lambda_smooth * smoothness_loss

# ========================================================================
# 3. Load and Prepare Data
# ========================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
all_files = sorted(list(dataset_dir.rglob("trial*_sync.npz")))

train_files = [f for f in all_files if f.parent.name == SOGGETTO_TARGET and f"trial{TRIAL_TEST_SELEZIONATO}" not in f.name]
test_file = [f for f in all_files if f.parent.name == SOGGETTO_TARGET and f"trial{TRIAL_TEST_SELEZIONATO}" in f.name][0]

print("Loading training data   ...")
train_seqs, train_targets = [], []
for f in train_files:
    seqs, tgts = load_and_prepare_data(f)
    train_seqs.extend(seqs)
    train_targets.extend(tgts)

print("Loading test data...")
test_seqs, test_targets = load_and_prepare_data(test_file)

X_train_tensor = torch.tensor(np.array(train_seqs), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(train_targets), dtype=torch.float32)
X_test_tensor = torch.tensor(np.array(test_seqs), dtype=torch.float32).to(device)
y_true_test = np.array(test_targets)

# ========================================================================
# 4. TRAINING LOOP
# ========================================================================
num_epochs = 50
batch_size = 64
learning_rate = 1e-3
weight_decay = 1e-4

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = SynergyLSTM(input_size=2, hidden_size=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = SmoothMSELoss(lambda_smooth=0.1)

print("\n TRAINING...")
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss Media: {epoch_loss/len(train_dataloader):.6f}")

save_dir.mkdir(parents=True, exist_ok=True)
model_path = save_dir / f'lstm_synergy_{SOGGETTO_TARGET}_trial{TRIAL_TEST_SELEZIONATO}.pt'
torch.save(model.state_dict(), model_path)
print(f"Model saved in: {model_path}")

# ========================================================================
# 5.INFERENCE
# ========================================================================
print("\n Performing Prediction ")
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).cpu().numpy()

# ========================================================================
# 6. SAVING CSV FOR UNITY AND PLOTTING
# ========================================================================
df_gt = convert_to_unity(y_true_test, fs=fs_omogenea)
df_pred = convert_to_unity(y_pred, fs=fs_omogenea)

csv_dir = save_dir / "unity_exports"
csv_dir.mkdir(exist_ok=True)

df_gt.to_csv(csv_dir / f"{SOGGETTO_TARGET}_trial{TRIAL_TEST_SELEZIONATO}_GT.csv", index=False)
df_pred.to_csv(csv_dir / f"{SOGGETTO_TARGET}_trial{TRIAL_TEST_SELEZIONATO}_Pred.csv", index=False)
print(f"CSV saved in: {csv_dir}")

# Plot
tempo = np.arange(len(y_true_test)) / fs_omogenea
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), sharey=True)
colors = plt.cm.tab20(np.arange(14))

for ax, data_plot, title in zip([ax1, ax2], [y_true_test, y_pred], ["TRUE (Ground Truth)", "PREDICTED"]):
    for ch in range(14):
        is_key = ch in [0, 4, 6, 8]
        ax.plot(tempo, data_plot[:, ch], color=colors[ch], linewidth=2.5 if is_key else 0.8, alpha=0.95 if is_key else 0.3)
    ax.set_title(title)
    ax.grid(True)
    ax.set_xlabel("Time (s)")
    if ax == ax1:
        ax.set_ylabel("Normalized Activation")

plt.tight_layout()
plt.show()